# Predicting the winning side from length and word frequency

**Does content-word frequency add predictive information beyond opening length?** All three models predict whether randomized side A won, using the same published training pairs and three OP-discussion folds. The inputs are signed **A minus B** differences.

| Model | Features | Script |
| --- | --- | --- |
| P0 | Opening word count | [train_p0_word_count_difference.py](train_p0_word_count_difference.py) |
| P1 | Mean content-word Zipf frequency | [train_p1_content_word_zipf_difference.py](train_p1_content_word_zipf_difference.py) |
| P2 | Both differences | [train_p2_word_count_and_content_word_zipf_differences.py](train_p2_word_count_and_content_word_zipf_differences.py) |

The word counts come from the shared pair assignments. Mean content-word Zipf uses the Minimal-cleaned openings, restricted to tokens tagged as nouns, verbs, adjectives, or adverbs. All models use L2 logistic regression with fixed `C=1`, and scaling is fitted inside each fold. The comparison reads timestamped outputs from the standalone scripts.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "team-setup").is_dir() and (p / "pyproject.toml").is_file()
)
ROOT = REPO / "project/data/processed/linguistic-features/paired-models"
MODELS = {
    "P0": "p0_word_count_difference",
    "P1": "p1_content_word_zipf_difference",
    "P2": "p2_word_count_and_content_word_zipf_differences",
}
RUN_IDS = {}  # Set an entry such as "P0": "timestamp" to revisit a run.
KEYS = ["pair_id", "post_id", "a_comment_id", "b_comment_id", "a_wins", "fold",
        "a_minus_b_word_count", "a_minus_b_mean_content_word_zipf"]

reports = {}
data = None
for label, folder in MODELS.items():
    runs = sorted((ROOT / folder).glob("*/results.json"))
    if label not in RUN_IDS and not runs:
        raise FileNotFoundError(f"Run train_{folder}.py to create model results.")
    report_path = ROOT / folder / RUN_IDS[label] / "results.json" if label in RUN_IDS else runs[-1]
    reports[label] = json.loads(report_path.read_text())
    frame = pd.read_parquet(report_path.parent / "oof_predictions.parquet")
    if data is None:
        data = frame[KEYS].copy()
    else:
        pd.testing.assert_frame_equal(data[KEYS], frame[KEYS])
    data[label] = frame["oof_probability_a_wins"]
if len({report["input_sha256"] for report in reports.values()}) != 1:
    raise ValueError("Choose three runs based on the same sensitivity input file.")
assert data.groupby("post_id")["fold"].nunique().eq(1).all()

summary = pd.DataFrame.from_dict(reports, orient="index")
display(summary[["features", "pairs", "discussions", "roc_auc", "average_precision", "a_win_prevalence"]].round(4))

,features,pairs,discussions,roc_auc,average_precision,a_win_prevalence
P0,[a_minus_b_word_count],3451,2507,0.6794,0.6684,0.4967
P1,[a_minus_b_mean_content_word_zipf],3451,2507,0.6154,0.6005,0.4967
P2,"[a_minus_b_word_count, a_minus_b_mean_content_word_zipf]",3451,2507,0.7032,0.6943,0.4967


## P2 evaluation

ROC-AUC measures how well P2 ranks A-wins above B-wins. Average precision summarizes precision across recall levels for A-wins. Pairwise accuracy chooses A when its predicted probability is at least 0.5 and B otherwise. Log loss evaluates those probabilities directly, with lower values indicating better probabilistic predictions. All four metrics use the same out-of-fold training pairs.

In [2]:
from sklearn.metrics import accuracy_score, average_precision_score, log_loss, roc_auc_score

p2_y = data["a_wins"]
p2_probability = data["P2"]
p2_evaluation = pd.DataFrame({
    "ROC-AUC": [roc_auc_score(p2_y, p2_probability)],
    "Average precision": [average_precision_score(p2_y, p2_probability)],
    "Pairwise accuracy": [accuracy_score(p2_y, p2_probability.ge(0.5))],
    "Log loss": [log_loss(p2_y, p2_probability)],
}, index=["P2"])
display(p2_evaluation.round(4))

,ROC-AUC,Average precision,Pairwise accuracy,Log loss
P2,0.7032,0.6943,0.6468,0.627


## Pearson correlation between model inputs

Calculate Pearson's \(r\) across the paired A minus B differences in word count and mean content-word Zipf frequency. The coefficient describes how the two inputs vary together linearly across pairs.

In [3]:
feature_columns = ["a_minus_b_word_count", "a_minus_b_mean_content_word_zipf"]
feature_values = data[feature_columns].dropna()
r = feature_values[feature_columns[0]].corr(feature_values[feature_columns[1]], method="pearson")
correlation = pd.DataFrame([{"Pairs": len(feature_values), "Pearson r": r}])
display(correlation.round(4))

,Pairs,Pearson r
0,3451,-0.0534


## Incremental performance

P2 minus P0 is the primary comparison. We estimate paired 95% intervals by resampling whole OP discussions and evaluating the two models on the same resampled rows. The out-of-fold scores are fixed for this calculation. The estimates describe predictive value in the paired training task. They do not show that changing a reply's vocabulary changes its chance of earning a delta.

In [4]:
from sklearn.metrics import average_precision_score, roc_auc_score

metric_functions = {"ROC-AUC": roc_auc_score, "Average precision": average_precision_score}
post_codes, posts = pd.factorize(data["post_id"])
rng = np.random.default_rng(42)
y = data["a_wins"].to_numpy()
p0, p2 = data[["P0", "P2"]].to_numpy().T
differences = {name: [] for name in metric_functions}
for _ in range(1000):
    post_weights = rng.multinomial(len(posts), np.full(len(posts), 1 / len(posts)))
    weights = post_weights[post_codes]
    for name, metric in metric_functions.items():
        differences[name].append(
            metric(y, p2, sample_weight=weights) - metric(y, p0, sample_weight=weights)
        )

increments = []
for name, metric in metric_functions.items():
    lo, hi = np.quantile(differences[name], [0.025, 0.975])
    increments.append({
        "Metric": name, "P2 minus P0": metric(y, p2) - metric(y, p0),
        "95% CI lower": lo, "95% CI upper": hi,
    })
incremental = pd.DataFrame(increments).set_index("Metric")
display(incremental.round(4))

,P2 minus P0,95% CI lower,95% CI upper
Metric,,,
ROC-AUC,0.0238,0.0098,0.037
Average precision,0.0259,0.0126,0.038


## Method verification

**Fold-specific scaling** keeps feature units from distorting L2 regularization. **Grouping by `post_id`** keeps a whole OP discussion together. **Seeded A/B assignment** gives the winner either position. **Resampling by discussion** keeps pairs from the same OP post together and uses the same weights for P0 and P2.

The published held-out pairs are reserved for a final generalization check for these models. The P0–P2 results here come from training cross-validation. Earlier project work examined held-out branch timing for a different task. The audit below checks the current paired models' inputs and scope.

In [5]:
import hashlib
import sys

sys.path.insert(0, str(REPO / "project/notebooks/linguistic-features"))
from paired_evaluation import SEED, make_paired_classifier

# Refit the exact shared pipeline and check each scaler against its fitting rows.
scalers_checked = 0
for label, report in reports.items():
    features = report["features"]
    for fold in sorted(data["fold"].unique()):
        validation = data["fold"].eq(fold)
        fitting = ~validation
        assert set(data.loc[fitting, "post_id"]).isdisjoint(data.loc[validation, "post_id"])
        train_features = data.loc[fitting, features]
        pipeline = make_paired_classifier().fit(train_features, data.loc[fitting, "a_wins"])
        scaler = pipeline.named_steps["standardscaler"]
        assert scaler.n_samples_seen_ == len(train_features)
        np.testing.assert_allclose(scaler.mean_, train_features.mean().to_numpy())
        np.testing.assert_allclose(scaler.var_, train_features.var(ddof=0).to_numpy())
        np.testing.assert_allclose(
            pipeline.predict_proba(data.loc[validation, features])[:, 1], data.loc[validation, label],
            rtol=1e-10, atol=1e-12,
        )
        scalers_checked += 1

# Reproduce the seeded assignments from the full pinned training-pair table.
assignments = pd.read_parquet(ROOT.parent / "training_pair_assignments.parquet").sort_values("pair_id")
expected_sides = np.random.default_rng(SEED).integers(0, 2, len(assignments))
np.testing.assert_array_equal(assignments["a_wins"].to_numpy(), expected_sides)
saved = assignments.set_index("pair_id").loc[data["pair_id"], ["post_id", "a_wins", "fold"]]
pd.testing.assert_frame_equal(data.set_index("pair_id")[saved.columns], saved)
a_win_count = int(data["a_wins"].sum())

# Each bootstrap draw assigns one multiplicity per discussion, shared by both models.
assert all(len(draws) == 1000 for draws in differences.values())
assert post_weights.sum() == len(posts)
assert pd.Series(weights).groupby(data["post_id"].to_numpy()).nunique().eq(1).all()

# Check the split of the exact source used by all three model runs.
for report in reports.values():
    source_dir = ROOT.parent / "word-frequency-sensitivity" / report["input_run"]
    assert report["split"] == json.loads((source_dir / "results.json").read_text())["split"] == "train"
    with (source_dir / "paired_scores.parquet").open("rb") as source:
        assert hashlib.file_digest(source, "sha256").hexdigest() == report["input_sha256"]

verification = pd.DataFrame([
    ("Standardization inside training folds", "Verified",
     f"{scalers_checked} fold-specific scalers reproduced the saved predictions."),
    ("Grouped cross-validation by OP discussion", "Verified",
     f"All {data['post_id'].nunique():,} discussions stayed in one validation fold."),
    ("Randomized A/B assignment", "Verified",
     f"Seed {SEED} reproduced the assignments. A won {a_win_count:,} of {len(data):,} pairs."),
    ("Bootstrap by discussion", "Verified",
     "1,000 paired draws with one weight per OP discussion."),
    ("Held-out generalization", "Pending final evaluation",
     "All three model inputs and scores in this notebook are from published training pairs."),
], columns=["Check", "Status", "Evidence"])
display(verification.set_index("Check"))

,Status,Evidence
Check,,
Standardization inside training folds,Verified,9 fold-specific scalers reproduced the saved predictions.
Grouped cross-validation by OP discussion,Verified,"All 2,507 discussions stayed in one validation fold."
Randomized A/B assignment,Verified,"Seed 42 reproduced the assignments. A won 1,714 of 3,451 pairs."
Bootstrap by discussion,Verified,"1,000 paired draws with one weight per OP discussion."
Held-out generalization,Pending final evaluation,All three model inputs and scores in this notebook are from published training pairs.
